In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from optbinning import BinningProcess

In [2]:
train_data = pd.read_csv('data/credit_scoring_data/train 2.csv')
test_data = pd.read_csv('data/credit_scoring_data/test 2.csv')

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_30105/1488526157.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('data/credit_scoring_data/train 2.csv')


In [3]:
train_data.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [4]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

In [5]:
train_data['Customer_ID'].nunique()

12500

In [6]:
train_data['Customer_ID'].unique()

array(['CUS_0xd40', 'CUS_0x21b1', 'CUS_0x2dbc', ..., 'CUS_0xaf61',
       'CUS_0x8600', 'CUS_0x942c'], shape=(12500,), dtype=object)

In [7]:
sample = train_data[train_data['Customer_ID'] == 'CUS_0x21b1']
sample

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
8,0x160e,CUS_0x21b1,January,Rick Rothackerj,28_,004-07-5839,_______,34847.84,3037.986667,2,...,Good,605.03,24.464031,26 Years and 7 Months,No,18.816215,104.291825168246,Low_spent_Small_value_payments,470.69062692529184,Standard
9,0x160f,CUS_0x21b1,February,Rick Rothackerj,28,004-07-5839,Teacher,34847.84,3037.986667,2,...,Good,605.03,38.550848,26 Years and 8 Months,No,18.816215,40.39123782853101,High_spent_Large_value_payments,484.5912142650067,Good
10,0x1610,CUS_0x21b1,March,Rick Rothackerj,28,004-07-5839,Teacher,34847.84_,3037.986667,2,...,_,605.03,33.224951,26 Years and 9 Months,No,18.816215,58.51597569589465,High_spent_Large_value_payments,466.46647639764313,Standard
11,0x1611,CUS_0x21b1,April,Rick Rothackerj,28,004-07-5839,Teacher,34847.84,NaN,2,...,Good,605.03,39.182656,26 Years and 10 Months,No,18.816215,99.30622796053305,Low_spent_Medium_value_payments,465.6762241330048,Good
12,0x1612,CUS_0x21b1,May,Rick Rothackerj,28,004-07-5839,Teacher,34847.84,3037.986667,2,...,Good,605.03,34.977895,26 Years and 11 Months,No,18.816215,130.11542024292334,Low_spent_Small_value_payments,444.8670318506144,Good
13,0x1613,CUS_0x21b1,June,Rick Rothackerj,28,004-07-5839,Teacher,34847.84,3037.986667,2,...,Good,605.03,33.381010,27 Years and 0 Months,No,18.816215,43.477190144355745,High_spent_Large_value_payments,481.505261949182,Good
14,0x1614,CUS_0x21b1,July,Rick Rothackerj,28,004-07-5839,Teacher,34847.84,NaN,2,...,Good,605.03,31.131702,27 Years and 1 Months,NM,18.816215,70.10177420755677,High_spent_Medium_value_payments,464.8806778859809,Good
15,0x1615,CUS_0x21b1,August,Rick Rothackerj,28,004-07-5839,Teacher,34847.84,3037.986667,2,...,Good,605.03,32.933856,27 Years and 2 Months,No,18.816215,218.90434353388733,Low_spent_Small_value_payments,356.07810855965045,Good


In [8]:
long_data = train_data.copy()

In [9]:
long_data = long_data.drop(columns = ['SSN', 'Name', 'ID'])

In [10]:
cat_cols = long_data.select_dtypes(include = ['object', 'string']).columns
cat_cols

Index(['Customer_ID', 'Month', 'Age', 'Occupation', 'Annual_Income',
       'Num_of_Loan', 'Type_of_Loan', 'Num_of_Delayed_Payment',
       'Changed_Credit_Limit', 'Credit_Mix', 'Outstanding_Debt',
       'Credit_History_Age', 'Payment_of_Min_Amount',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='object')

In [11]:
long_data['Credit_Score'].value_counts()

Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64

In [12]:
# str_to_num = ['Age', 'Annual_Income', 'Num_of_Loan','Num_of_Delayed_Payment', 'Credit_History_Age', 
#               'Amount_invested_monthly', 'Monthly_Balance', 'month2_balance', 'month3_balance',
#               'Changed_Credit_Limit', 'Outstanding_Debt']

In [13]:
long_data['Credit_History_Age'].unique()

array(['22 Years and 1 Months', nan, '22 Years and 3 Months',
       '22 Years and 4 Months', '22 Years and 5 Months',
       '22 Years and 6 Months', '22 Years and 7 Months',
       '26 Years and 7 Months', '26 Years and 8 Months',
       '26 Years and 9 Months', '26 Years and 10 Months',
       '26 Years and 11 Months', '27 Years and 0 Months',
       '27 Years and 1 Months', '27 Years and 2 Months',
       '17 Years and 9 Months', '17 Years and 10 Months',
       '17 Years and 11 Months', '18 Years and 1 Months',
       '18 Years and 2 Months', '18 Years and 3 Months',
       '18 Years and 4 Months', '17 Years and 3 Months',
       '17 Years and 4 Months', '17 Years and 5 Months',
       '17 Years and 6 Months', '17 Years and 7 Months',
       '17 Years and 8 Months', '30 Years and 8 Months',
       '30 Years and 9 Months', '30 Years and 10 Months',
       '30 Years and 11 Months', '31 Years and 0 Months',
       '31 Years and 1 Months', '31 Years and 2 Months',
       '31 Years and

In [14]:
import re

def period_to_months(row):
    if pd.isna(row):
        return 0
    years = re.search(r'(\d+)\s*Year', row)
    months = re.search(r'(\d+)\s*Month', row)
    num_years = int(years.group(1)) if years else 0
    num_months = int(months.group(1)) if months else 0
    total_months = (num_years * 12 )+ num_months
    return int(total_months)


In [15]:
long_data['Credit_History_Age'] = long_data['Credit_History_Age'].apply(period_to_months)

In [16]:
long_data['Credit_History_Age'].dtype

dtype('int64')

In [17]:
long_data['Credit_History_Age'][:10]

0    265
1      0
2    267
3    268
4    269
5    270
6    271
7      0
8    319
9    320
Name: Credit_History_Age, dtype: int64

In [18]:
# long_data['Credit_History_Age'] = long_data['Credit_History_Age'].str.strip('.').astype(float)

In [19]:
# long_data['Credit_History_Age'].unique()
str_to_num = ['Age', 'Annual_Income', 'Num_of_Loan','Num_of_Delayed_Payment',
              'Amount_invested_monthly', 'Monthly_Balance',
            #   , 'month2_balance', 'month3_balance',
              'Changed_Credit_Limit', 'Outstanding_Debt']

In [20]:
long_data['Outstanding_Debt'].unique()

array(['809.98', '605.03', '1303.01', ..., '3571.7_', '3571.7', '502.38'],
      shape=(13178,), dtype=object)

In [21]:
long_data[str_to_num] = long_data[str_to_num].apply(
    lambda col: pd.to_numeric(
        col.astype(str).str.replace('_', '', regex=False),
        errors='coerce'
    )
)

In [22]:
long_data['Annual_Income'].unique()

array([ 19114.12,  34847.84, 143162.64, ...,  37188.1 ,  20002.88,
        39628.99], shape=(13487,))

In [23]:
long_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Customer_ID               100000 non-null  object 
 1   Month                     100000 non-null  object 
 2   Age                       100000 non-null  int64  
 3   Occupation                100000 non-null  object 
 4   Annual_Income             100000 non-null  float64
 5   Monthly_Inhand_Salary     84998 non-null   float64
 6   Num_Bank_Accounts         100000 non-null  int64  
 7   Num_Credit_Card           100000 non-null  int64  
 8   Interest_Rate             100000 non-null  int64  
 9   Num_of_Loan               100000 non-null  int64  
 10  Type_of_Loan              88592 non-null   object 
 11  Delay_from_due_date       100000 non-null  int64  
 12  Num_of_Delayed_Payment    92998 non-null   float64
 13  Changed_Credit_Limit      97909 non-null   fl

In [24]:
cat_cols = long_data.select_dtypes(include = ['object','string']).columns

## Feature Engineering

In [25]:
cat_cols

Index(['Customer_ID', 'Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix',
       'Payment_of_Min_Amount', 'Payment_Behaviour', 'Credit_Score'],
      dtype='object')

In [26]:
long_data['Type_of_Loan'] = long_data['Type_of_Loan'].str.replace('and', '', regex=False)

In [27]:
long_data['Type_of_Loan'].unique().tolist()

['Auto Loan, Credit-Builder Loan, Personal Loan,  Home Equity Loan',
 'Credit-Builder Loan',
 'Auto Loan, Auto Loan,  Not Specified',
 'Not Specified',
 nan,
 'Credit-Builder Loan,  Mortgage Loan',
 'Not Specified, Auto Loan,  Student Loan',
 'Personal Loan, Debt Consolidation Loan,  Auto Loan',
 'Not Specified,  Payday Loan',
 'Credit-Builder Loan, Personal Loan,  Auto Loan',
 'Payday Loan,  Payday Loan',
 'Not Specified, Student Loan,  Personal Loan',
 'Personal Loan, Payday Loan, Student Loan, Auto Loan, Home Equity Loan, Student Loan,  Payday Loan',
 'Not Specified, Student Loan, Student Loan, Credit-Builder Loan,  Auto Loan',
 'Payday Loan,  Home Equity Loan',
 'Credit-Builder Loan, Not Specified, Mortgage Loan, Payday Loan, Credit-Builder Loan,  Personal Loan',
 'Mortgage Loan, Debt Consolidation Loan, Payday Loan, Auto Loan,  Not Specified',
 'Credit-Builder Loan, Mortgage Loan, Mortgage Loan, Credit-Builder Loan,  Student Loan',
 'Not Specified, Student Loan,  Student Loan',
 '

In [28]:
unique_loans = long_data['Type_of_Loan'].str.split(',').explode().str.strip().unique().tolist()

In [29]:
unique_loans

['Auto Loan',
 'Credit-Builder Loan',
 'Personal Loan',
 'Home Equity Loan',
 'Not Specified',
 nan,
 'Mortgage Loan',
 'Student Loan',
 'Debt Consolidation Loan',
 'Payday Loan']

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Add any other comma-separated text columns here
# tfidf_cols = ['Type_of_Loan']

tfidf_dfs = []

# for col in tfidf_cols:
docs = (
    long_data['Type_of_Loan']
    .fillna('')
    .astype(str)
    .str.split(',')
    .apply(lambda items: ' '.join(
        item.strip().lower().replace(' ', '_')
        for item in items if item.strip()
    ))
)


In [31]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f"loan_type__{feat}" for feat in vectorizer.get_feature_names_out()],
    index=long_data.index
)
tfidf_dfs.append(tfidf_df)

In [32]:
# long_data = long_data.drop(columns=tfidf_cols)
new_data = pd.concat([long_data] + tfidf_dfs, axis=1)
new_data.shape

(100000, 35)

In [33]:
new_data.head()

,Customer_ID,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,...,loan_type__auto_loan,loan_type__builder_loan,loan_type__credit,loan_type__debt_consolidation_loan,loan_type__home_equity_loan,loan_type__mortgage_loan,loan_type__not_specified,loan_type__payday_loan,loan_type__personal_loan,loan_type__student_loan
0,CUS_0xd40,January,23,Scientist,19114.12,1824.843333,3,4,3,4,...,0.45216,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0
1,CUS_0xd40,February,23,Scientist,19114.12,NaN,3,4,3,4,...,0.45216,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0
2,CUS_0xd40,March,-500,Scientist,19114.12,NaN,3,4,3,4,...,0.45216,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0
3,CUS_0xd40,April,23,Scientist,19114.12,NaN,3,4,3,4,...,0.45216,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0
4,CUS_0xd40,May,23,Scientist,19114.12,1824.843333,3,4,3,4,...,0.45216,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0


In [34]:
cat_cols

Index(['Customer_ID', 'Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix',
       'Payment_of_Min_Amount', 'Payment_Behaviour', 'Credit_Score'],
      dtype='object')

In [35]:
long_data['Occupation'].unique()

array(['Scientist', '_______', 'Teacher', 'Engineer', 'Entrepreneur',
       'Developer', 'Lawyer', 'Media_Manager', 'Doctor', 'Journalist',
       'Manager', 'Accountant', 'Musician', 'Mechanic', 'Writer',
       'Architect'], dtype=object)

In [36]:
new_data = new_data.drop(columns = ['Type_of_Loan'])

In [37]:
cat_cols_new = new_data.select_dtypes(include = ['object', 'string']).columns.tolist()
cat_cols_new

['Customer_ID',
 'Month',
 'Occupation',
 'Credit_Mix',
 'Payment_of_Min_Amount',
 'Payment_Behaviour',
 'Credit_Score']

In [38]:
cat_cols_new.remove('Customer_ID')

In [39]:
cat_columns = [col for col in cat_cols_new if col != 'Target']
cat_columns

['Month',
 'Occupation',
 'Credit_Mix',
 'Payment_of_Min_Amount',
 'Payment_Behaviour',
 'Credit_Score']

In [40]:
# new_data['Credit_History_Age'].unique()

One-hot encoding categorical columns

In [41]:
new_data['mix_target'] = new_data['Credit_Score'] + new_data['Credit_Mix']
new_data['mix_target'].unique()

array(['Good_', 'GoodGood', 'StandardGood', 'Standard_',
       'StandardStandard', 'PoorStandard', 'Poor_', 'GoodStandard',
       'PoorBad', 'StandardBad', 'GoodBad', 'PoorGood'], dtype=object)

In [42]:
new_data['mix_target'].value_counts()

mix_target
StandardStandard    26577
GoodGood            11875
PoorBad             11409
Standard_           10704
StandardGood         8601
PoorStandard         7859
StandardBad          7292
Poor_                5869
PoorGood             3861
Good_                3622
GoodStandard         2043
GoodBad               288
Name: count, dtype: int64

In [43]:
new_data['Credit_Score'].value_counts()

Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64

the credit mix statuses can be very different from the actual credit standing of the customer


In [44]:
new_data.head()

,Customer_ID,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,...,loan_type__builder_loan,loan_type__credit,loan_type__debt_consolidation_loan,loan_type__home_equity_loan,loan_type__mortgage_loan,loan_type__not_specified,loan_type__payday_loan,loan_type__personal_loan,loan_type__student_loan,mix_target
0,CUS_0xd40,January,23,Scientist,19114.12,1824.843333,3,4,3,4,...,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0,Good_
1,CUS_0xd40,February,23,Scientist,19114.12,NaN,3,4,3,4,...,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0,GoodGood
2,CUS_0xd40,March,-500,Scientist,19114.12,NaN,3,4,3,4,...,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0,GoodGood
3,CUS_0xd40,April,23,Scientist,19114.12,NaN,3,4,3,4,...,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0,GoodGood
4,CUS_0xd40,May,23,Scientist,19114.12,1824.843333,3,4,3,4,...,0.4444,0.4444,0.0,0.44655,0.0,0.0,0.0,0.44851,0.0,GoodGood


In [45]:
data = new_data.copy()

In [46]:
data['Month'] = pd.to_datetime(new_data['Month'], format='%B').dt.month
# data.head()
data['order'] = data.sort_values(by='Month', ascending= False).groupby('Customer_ID').cumcount() + 1
# data[['Customer_ID', 'Month', 'order']].head(20)
# records  = data[data['order'] <= 3]
# records.head()

In [47]:
features = data.groupby('Customer_ID').agg(
    max_delayed_payment = ('Num_of_Delayed_Payment', 'max'),
    max_credit_util = ('Credit_Utilization_Ratio', 'max'),
    credit_util_var = ('Credit_Utilization_Ratio', 'var')
).reset_index()
features.head()

,Customer_ID,max_delayed_payment,max_credit_util,credit_util_var
0,CUS_0x1000,28.0,40.082272,23.612754
1,CUS_0x1009,1749.0,40.286997,28.889929
2,CUS_0x100b,9.0,43.829630,30.524752
3,CUS_0x1011,17.0,29.198639,1.677459
4,CUS_0x1013,9.0,41.920614,42.617492


In [48]:
month1 = data[data['order'] == 1].rename(
    columns= {'Month': 'latest_month', 'Credit_Score': 'Target'})
# month1.head()
month2 = data[data['order'] == 2][['Customer_ID', 'Monthly_Balance', 'Credit_Utilization_Ratio', 'Num_of_Delayed_Payment', 'Delay_from_due_date']].rename(
    columns= {'Monthly_Balance': 'month2_balance', 'Credit_Utilization_Ratio': 'month2_credit_util', 'Num_of_Delayed_Payment': 'month2_delayed_payment', 'Delay_from_due_date': 'month2_delay_from_due_date'})
month3 = data[data['order'] == 3][['Customer_ID', 'Monthly_Balance', 'Credit_Utilization_Ratio', 'Num_of_Delayed_Payment', 'Delay_from_due_date']].rename(
    columns= {'Monthly_Balance': 'month3_balance', 'Credit_Utilization_Ratio': 'month3_credit_util', 'Num_of_Delayed_Payment': 'month3_delayed_payment', 'Delay_from_due_date': 'month3_delay_from_due_date'})
user_history = month1.merge(month2, on= 'Customer_ID', how = 'left').merge(
    month3 , on = 'Customer_ID', how = 'left'
    ).merge(features, on = 'Customer_ID', how = 'left')
user_history.head()

,Customer_ID,latest_month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,...,month2_credit_util,month2_delayed_payment,month2_delay_from_due_date,month3_balance,month3_credit_util,month3_delayed_payment,month3_delay_from_due_date,max_delayed_payment,max_credit_util,credit_util_var
0,CUS_0xd40,8,23,Scientist,19114.12,1824.843333,3,4,3,4,...,22.537593,8.0,3,340.479212,27.262259,4.0,8,8.0,31.944960,11.466897
1,CUS_0x21b1,8,28,Teacher,34847.84,3037.986667,2,4,6,1,...,31.131702,4.0,3,481.505262,33.381010,0.0,3,4.0,39.182656,21.093257
2,CUS_0x2dbc,8,34,Engineer,143162.64,12187.220000,1,5,8,3,...,38.068624,6.0,8,963.921581,39.783993,6.0,8,8.0,41.702573,33.246935
3,CUS_0xb891,8,55,Entrepreneur,30689.89,2612.490833,2,5,4,-100,...,26.056395,NaN,5,419.880784,27.445422,6.0,5,9.0,41.154317,34.213323
4,CUS_0x1cdb,8,21,Developer,35547.71,2853.309167,7,5,5,-100,...,26.263823,15.0,10,497.687279,29.217556,15.0,5,17.0,41.776187,45.494381


In [49]:
user_history['Num_of_Loan'] = user_history['Num_of_Loan'].abs()

In [50]:
user_history['Num_of_Loan'].min()

np.int64(0)

In [51]:
user_history['balance_m1_m2'] = (user_history['Monthly_Balance'] - user_history['month2_balance']) / user_history['month2_balance']
user_history['balance_m2_m3'] = (user_history['month2_balance'] - user_history['month3_balance']) / user_history['month3_balance']

In [52]:
model_data = user_history.drop(columns = ['Customer_ID', 'order', 'mix_target', 'latest_month'])

In [53]:
cat_features = model_data.select_dtypes(include = ['object', 'string']).columns.tolist()
cat_features

['Occupation',
 'Credit_Mix',
 'Payment_of_Min_Amount',
 'Payment_Behaviour',
 'Target']

In [54]:
cat_features.remove('Target')

In [55]:
for col in cat_features:
    model_data[col] = model_data[col].astype('category')
target_map = {'Poor': 0, 'Standard': 1, 'Good': 2}
model_data['Target'] = model_data['Target'].map(target_map)


## LGBM Credit Classification Model

In [56]:
X = model_data.drop(columns = ['Target'])
y = model_data['Target']
X_train, X_test , y_train , y_test = train_test_split(X,y, test_size = 0.25, 
                                                      random_state=42, stratify = y)


In [57]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# y_train should be your training labels
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# For LightGBM
# lgbm = LGBMClassifier(class_weight=class_weight_dict, ...)



In [58]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

train_set = lgb.Dataset(X_train, label=y_train, weight=sample_weights,
                        categorical_feature=cat_features)
test_set = lgb.Dataset(X_test, label=y_test, reference=train_set,
                       categorical_feature=cat_features)

In [59]:
train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
val_set = lgb.Dataset(X_test, label=y_test, reference = train_set)

In [60]:
params = {
    'objective':        'multiclass',
    'num_class':        3,
    'metric':           'multi_logloss',
    'num_leaves':       63,
    'learning_rate':    0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'verbose':          -1,
    'is_unbalance':     True
    # 'class_weight':     class_weight_dict
}

In [61]:
model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[val_set],
    # callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)

In [62]:
model.feature_importance(importance_type='gain')

array([ 3543.3913901 ,  4332.26770306,  3577.98831941,  6007.33018756,
        2711.71767171,  3794.73020206,  8580.18496111,  1565.03979905,
        3215.26248213,  3814.44216075,  8085.8062278 ,  3995.38805661,
        6187.7066435 , 14816.65747655,  4743.07059454,  5551.77976115,
        2558.23592891,  6199.65836079,  7255.28701715,  1682.12512301,
        4733.03083069,  1667.31808302,  1532.76231062,   339.08160965,
        1877.37617405,  1767.21922392,  1871.40455845,  1574.01489466,
        1827.95271818,  1696.34921134,  1871.90557468,  4664.20433006,
        5176.86132538,  3569.10016407,  3132.4180097 ,  4921.77992545,
        5036.1674239 ,  3781.11154167,  3994.72206452,  2282.57555325,
        4392.74538956,  4910.12365101,  5535.40352486,  5214.77607342])

In [63]:
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import cohen_kappa_score, roc_auc_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [64]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

In [65]:
from lightgbm import LGBMClassifier

In [66]:
## Cross validation
def lgb_objective(trial, X_train, X_val , y_train , y_val):
    params = {
        'objective':           'multiclass',
        'num_class':           3,
        'metric':              'multi_logloss',
        'n_estimators':        1000,
        # 'early_stopping_rounds': 50,
        'random_state':        42,
        # 'verbose':             -1,
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves':          trial.suggest_int('num_leaves', 10, 200),
        'max_depth':           trial.suggest_int('max_depth', 3, 10),
        'subsample':           trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':    trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples':   trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha':           trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'cat_l2': trial.suggest_float('cat_l2', 1, 30),
        'cat_smooth': trial.suggest_float('cat_smooth', 1, 50),
        'reg_lambda':          trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    model = LGBMClassifier(**params, class_weight=class_weight_dict)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=cat_features,
        eval_metric='multi_logloss',
        # early_stopping_rounds=30,
        # verbose=False
    )
    
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, X_train, X_test, y_train, y_test), n_trials=30)

print("Best params:", study.best_params)

Best params: {'learning_rate': 0.029043168600341025, 'num_leaves': 143, 'max_depth': 8, 'subsample': 0.9470923311119088, 'colsample_bytree': 0.9554669633041123, 'min_child_samples': 84, 'reg_alpha': 0.13802857293602833, 'cat_l2': 12.59920093356147, 'cat_smooth': 12.878567377973717, 'reg_lambda': 0.03281645129230756}


In [67]:
opt_lgb_model = lgb.LGBMClassifier(**study.best_params)
opt_lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], categorical_feature=cat_features, eval_metric='multi_logloss')


,boosting_type,'gbdt'
,num_leaves,143
,max_depth,8
,learning_rate,0.029043168600341025
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,84


### LGBM Evaluation

In [68]:
# lgb_study = optuna.create_study(direction='maximize')
# lgb_study.optimize(lambda trial: lgb_objective(trial, X, y), n_trials=50)
y_prob = opt_lgb_model.predict_proba(X_test)   # shape (n_samples, 3) — then argmax makes sense
y_pred = opt_lgb_model.predict(X_test)          # shape (n_samples,) — class labels directly


qwk = cohen_kappa_score(y_test, y_pred, weights='quadratic')
print(f"QWK: {qwk:.4f}")
print(classification_report(y_test, y_pred, target_names=['Poor', 'Standard', 'Good']))
print(confusion_matrix(y_test, y_pred))

# print(f"Best LGB | QWK: {lgb_study.best_value:.4f} | Params: {lgb_study.best_params}")


QWK: 0.5043
              precision    recall  f1-score   support

        Poor       0.64      0.55      0.59       901
    Standard       0.63      0.75      0.68      1621
        Good       0.58      0.40      0.47       603

    accuracy                           0.62      3125
   macro avg       0.62      0.57      0.58      3125
weighted avg       0.62      0.62      0.62      3125

[[ 496  374   31]
 [ 264 1212  145]
 [  10  353  240]]


In [69]:
auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
print(f"AUC: {auc:.4f}")

AUC: 0.7822


In [70]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
# import optuna
# import numpy as np

def lgb_objective(trial, X, y):
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'n_estimators': 1000,
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'cat_l2': trial.suggest_float('cat_l2', 1, 30),
        'cat_smooth': trial.suggest_float('cat_smooth', 1, 50),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }

    # Example target size strategy:
    # - cap majority classes by undersampling
    # - raise minority classes by oversampling
    under = RandomUnderSampler(
        sampling_strategy={
            0: 500,   # class 0 down to 1500
            1: 500,   # class 1 down to 1500
            2: 300    # keep / downsample class 2 if needed
        },
        random_state=42
    )

    over = RandomOverSampler(
        sampling_strategy={
            2: 500    # oversample minority class 2 up to 500
        },
        random_state=42
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X, y):
        X_train_fold = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
        X_val_fold = X.iloc[val_idx] if hasattr(X, "iloc") else X[val_idx]
        y_train_fold = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
        y_val_fold = y.iloc[val_idx] if hasattr(y, "iloc") else y[val_idx]

        # Resample ONLY training fold
        X_res, y_res = under.fit_resample(X_train_fold, y_train_fold)
        X_res, y_res = over.fit_resample(X_res, y_res)

        model = LGBMClassifier(**params)

        model.fit(
            X_res,
            y_res,
            eval_set=[(X_val_fold, y_val_fold)],
            categorical_feature=cat_features,
            eval_metric='multi_logloss'
        )

        preds = model.predict(X_val_fold)
        scores.append(accuracy_score(y_val_fold, preds))

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, X_train, y_train), n_trials=30)

print("Best params:", study.best_params)

Best params: {'learning_rate': 0.033457606596029504, 'num_leaves': 181, 'max_depth': 10, 'subsample': 0.9943690510825496, 'colsample_bytree': 0.566487510381879, 'min_child_samples': 10, 'reg_alpha': 1.4374220740823367, 'cat_l2': 26.472361615480125, 'cat_smooth': 47.12483450847602, 'reg_lambda': 0.17745071225387926}


In [71]:
optsample_lgb_model = lgb.LGBMClassifier(**study.best_params)
optsample_lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], categorical_feature=cat_features, eval_metric='multi_logloss')

# lgb_study = optuna.create_study(direction='maximize')
# lgb_study.optimize(lambda trial: lgb_objective(trial, X, y), n_trials=50)
y_prob = optsample_lgb_model.predict_proba(X_test)   # shape (n_samples, 3) — then argmax makes sense
y_pred = optsample_lgb_model.predict(X_test)          # shape (n_samples,) — class labels directly


qwk = cohen_kappa_score(y_test, y_pred, weights='quadratic')
print(f"QWK: {qwk:.4f}")
print(classification_report(y_test, y_pred, target_names=['Poor', 'Standard', 'Good']))
print(confusion_matrix(y_test, y_pred))

# print(f"Best LGB | QWK: {lgb_study.best_value:.4f} | Params: {lgb_study.best_params}")

auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
print(f"AUC: {auc:.4f}")

QWK: 0.5124
              precision    recall  f1-score   support

        Poor       0.65      0.56      0.60       901
    Standard       0.63      0.75      0.68      1621
        Good       0.57      0.41      0.48       603

    accuracy                           0.63      3125
   macro avg       0.62      0.57      0.59      3125
weighted avg       0.62      0.63      0.62      3125

[[ 505  365   31]
 [ 261 1209  151]
 [  10  347  246]]
AUC: 0.7824


## XGBoost Model

In [72]:
from xgboost import XGBClassifier

In [78]:
## Cross validation
def xgb_objective(trial, X_train, X_val , y_train , y_val):
    xgb_params = {
        'max_cat_to_onehot': 7,
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'max_cat_threshold': trial.suggest_int('max_cat_threshold', 1, 64),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-4, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        'tree_method': trial.suggest_categorical('tree_method', [ 'hist']),
        'max_leaves': trial.suggest_int('max_leaves', 10, 200),
        'max_bin': trial.suggest_int('max_bin', 256, 1024),
        
    }
    sample_weights = y_train.map(class_weight_dict)
    model = XGBClassifier(**xgb_params, enable_categorical = True,
        eval_metric = 'mlogloss')
    model.fit(
        X_train, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_val, y_val)],
        # early_stopping_rounds=30,
        # verbose=False
    )
    
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: xgb_objective(trial, X_train, X_test, y_train, y_test), n_trials=30)

print("Best params:", study.best_params)

[0]	validation_0-mlogloss:1.07187
[1]	validation_0-mlogloss:1.04932
[2]	validation_0-mlogloss:1.03016
[3]	validation_0-mlogloss:1.01231
[4]	validation_0-mlogloss:0.99731
[5]	validation_0-mlogloss:0.98342
[6]	validation_0-mlogloss:0.97151
[7]	validation_0-mlogloss:0.96140
[8]	validation_0-mlogloss:0.95252
[9]	validation_0-mlogloss:0.94381
[10]	validation_0-mlogloss:0.93543
[11]	validation_0-mlogloss:0.92819
[12]	validation_0-mlogloss:0.92238
[13]	validation_0-mlogloss:0.91653
[14]	validation_0-mlogloss:0.91140
[15]	validation_0-mlogloss:0.90668
[16]	validation_0-mlogloss:0.90255
[17]	validation_0-mlogloss:0.89893
[18]	validation_0-mlogloss:0.89460
[19]	validation_0-mlogloss:0.89041
[20]	validation_0-mlogloss:0.88731
[21]	validation_0-mlogloss:0.88441
[22]	validation_0-mlogloss:0.88154
[23]	validation_0-mlogloss:0.87838
[24]	validation_0-mlogloss:0.87575
[25]	validation_0-mlogloss:0.87376
[26]	validation_0-mlogloss:0.87188
[27]	validation_0-mlogloss:0.87006
[28]	validation_0-mlogloss:0.8

In [79]:
opt_xgb_model = XGBClassifier(**study.best_params, eval_metric='mlogloss', enable_categorical=True)
opt_xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])


[0]	validation_0-mlogloss:0.99208
[1]	validation_0-mlogloss:0.97146
[2]	validation_0-mlogloss:0.95320
[3]	validation_0-mlogloss:0.93810
[4]	validation_0-mlogloss:0.92444
[5]	validation_0-mlogloss:0.91284
[6]	validation_0-mlogloss:0.90247
[7]	validation_0-mlogloss:0.89359
[8]	validation_0-mlogloss:0.88522
[9]	validation_0-mlogloss:0.87812
[10]	validation_0-mlogloss:0.87075
[11]	validation_0-mlogloss:0.86489
[12]	validation_0-mlogloss:0.85936
[13]	validation_0-mlogloss:0.85463
[14]	validation_0-mlogloss:0.84991
[15]	validation_0-mlogloss:0.84574
[16]	validation_0-mlogloss:0.84179
[17]	validation_0-mlogloss:0.83833
[18]	validation_0-mlogloss:0.83516
[19]	validation_0-mlogloss:0.83297
[20]	validation_0-mlogloss:0.83010
[21]	validation_0-mlogloss:0.82822
[22]	validation_0-mlogloss:0.82606
[23]	validation_0-mlogloss:0.82428
[24]	validation_0-mlogloss:0.82256
[25]	validation_0-mlogloss:0.82101
[26]	validation_0-mlogloss:0.82001
[27]	validation_0-mlogloss:0.81859
[28]	validation_0-mlogloss:0.8

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [80]:
y_prob = opt_xgb_model.predict_proba(X_test)   # shape (n_samples, 3) — then argmax makes sense
y_pred = opt_xgb_model.predict(X_test)          # shape (n_samples,) — class labels directly


qwk = cohen_kappa_score(y_test, y_pred, weights='quadratic')
print(f"QWK: {qwk:.4f}")
print(classification_report(y_test, y_pred, target_names=['Poor', 'Standard', 'Good']))
print(confusion_matrix(y_test, y_pred))



QWK: 0.5000
              precision    recall  f1-score   support

        Poor       0.63      0.54      0.59       901
    Standard       0.63      0.74      0.68      1621
        Good       0.57      0.43      0.49       603

    accuracy                           0.62      3125
   macro avg       0.61      0.57      0.59      3125
weighted avg       0.62      0.62      0.62      3125

[[ 490  374   37]
 [ 269 1196  156]
 [  15  328  260]]


In [81]:
auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
print(f"AUC: {auc:.4f}")

AUC: 0.7784


In [ ]:
# from sklearn.calibration import calibration_curve
# import matplotlib.pyplot as plt

# for i, label in enumerate(['Poor', 'Standard', 'Good']):
#     prob_true, prob_pred = calibration_curve(
#         (y_test == i).astype(int), probs[:, i], n_bins=10
#     )
#     plt.plot(prob_pred, prob_true, marker='o', label=label)

# plt.plot([0, 1], [0, 1], '--', color='grey', label='Perfect')
# plt.xlabel('Mean predicted probability')
# plt.ylabel('Fraction of positives')
# plt.legend()
# plt.title('Calibration curves')
# plt.show()


In [ ]:
# importances = final_xgb.feature_importances_   # or model.feature_importance(importance_type='gain') for LightGBM
# feature_names = X.columns

# importance_df = pd.DataFrame({
#     'feature': feature_names,
#     'importance': importances
# }).sort_values('importance', ascending=False)

# # Normalize to sum to 100 for easy reading
# importance_df['importance_pct'] = 100 * importance_df['importance'] / importance_df['importance'].sum()

# print(importance_df.head(20).to_string(index=False))


## Deep Learning - MultiClass Classification

In [ ]:
data_dl = user_history.drop(columns = ['Customer_ID', 'order', 'mix_target', 'latest_month'])

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.preprocessing import StandardScaler

In [ ]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
num_cols = data_dl.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = data_dl.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

In [ ]:
cat_cols.remove('Target')
cat_cols

['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

In [ ]:
# Ordinal encoding — preserve the natural order
label_map = {"Poor": 0, "Standard": 1, "Good": 2}
data_dl["Target_label"] = data_dl["Target"].map(label_map)

In [ ]:
target_cols = ['Target_label']

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight= 'balanced',
    classes=np.array([0, 1, 2]),
    y=data_dl["Target_label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
# Pass into loss function — model pays more attention to rare classes
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
X_train, X_test , y_train , y_test = train_test_split(
    data_dl.drop(columns=target_cols),
    data_dl[target_cols],
    test_size=0.25,
    random_state=42,
    stratify=data_dl[target_cols]
)

In [ ]:
label_encoder = LabelEncoder().fit(data_dl[cat_cols].values.ravel())
scaler = StandardScaler().fit(X_train[num_cols])

In [ ]:
## Creating Custom DataSet
class HistDataset(Dataset):
    def __init__(self, df, cat_cols, cont_cols, label_col, scaler, label_encoder):
        self.cat_data = df[cat_cols].copy()
        self.cont_data = df[cont_cols].copy()
        self.labels = df[label_col].values

        # Apply label encoders
        for col in cat_cols:
            self.cat_data[col] = label_encoder.transform(self.cat_data[col].astype(str))

        # Apply scaler
        self.cont_data = scaler.transform(self.cont_data)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        cat = torch.tensor(self.cat_data.iloc[idx].values, dtype=torch.long)
        cont = torch.tensor(self.cont_data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return cat, cont, label

In [ ]:
train_set = HistDataset(X_train, cat_cols, num_cols, 'Target_label', scaler, label_encoder)
val_set = HistDataset(X_test, cat_cols, num_cols, 'Target_label', scaler, label_encoder)